# 04 — Gold: star schema + serving views

Shape the silver tables into a small **star schema** plus a couple of pre-aggregated views
for BI/Genie-style consumption.

```
        dim_date                         dim_team
           |                              |     |
           |                       home __|     |__ away
           v                              v     v
        fact_games  <-----------------  (one row per game)
           ^
           |  game_sk
        fact_team_game  (one row per team per game, Four Factors + ratings)
           ^   |
           team_sk -> dim_team
```

`RELY` primary/foreign keys give the optimizer star-schema hints and render an ERD in
Catalog Explorer. Facts are `CLUSTER BY (season, game_date)`.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

from nba_helpers import ABBREVIATION_METADATA, CURRENT_TEAM_ABBREVIATIONS

load_dotenv()
spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA",  "basketball_reference_waf")
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"

GAMES    = f"{UC_CATALOG}.{SILVER_SCHEMA}.games"
TEAM_BOX = f"{UC_CATALOG}.{SILVER_SCHEMA}.team_game_box"
DIM_TEAM = f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_team"
DIM_DATE = f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_date"
FACT_GAMES = f"{UC_CATALOG}.{GOLD_SCHEMA}.fact_games"
FACT_TG    = f"{UC_CATALOG}.{GOLD_SCHEMA}.fact_team_game"

def add_pk(table, name, cols):
    for c in cols:
        try: spark.sql(f"ALTER TABLE {table} ALTER COLUMN {c} SET NOT NULL")
        except Exception as e:
            if "ALREADY" not in str(e).upper(): raise
    try:
        spark.sql(f"ALTER TABLE {table} DROP CONSTRAINT IF EXISTS {name}")
        spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} PRIMARY KEY ({', '.join(cols)}) RELY")
    except Exception as e: print(f"  PK {name}: {str(e)[:80]}")

def add_fk(table, name, cols, ref_table, ref_cols):
    try:
        spark.sql(f"ALTER TABLE {table} DROP CONSTRAINT IF EXISTS {name}")
        spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} FOREIGN KEY ({', '.join(cols)}) "
                  f"REFERENCES {ref_table} ({', '.join(ref_cols)}) RELY")
    except Exception as e: print(f"  FK {name}: {str(e)[:80]}")

print("Gold targets ready.")

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Gold targets ready.


## `dim_team` — from the helper metadata (abbreviation, name, conference, division)

In [2]:
rows = [(a, ABBREVIATION_METADATA[a][0], ABBREVIATION_METADATA[a][1], ABBREVIATION_METADATA[a][2])
        for a in CURRENT_TEAM_ABBREVIATIONS]
spark.createDataFrame(rows, ["team_id", "team_name", "conference", "division"]).createOrReplaceTempView("team_lookup")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DIM_TEAM} (
        team_sk    STRING NOT NULL COMMENT 'MD5(team_id)',
        team_id    STRING COMMENT 'Basketball Reference team abbreviation',
        team_name  STRING,
        conference STRING,
        division   STRING
    ) COMMENT 'Gold dimension - the 30 current NBA franchises.'
""")
spark.sql(f"""
    INSERT OVERWRITE {DIM_TEAM}
    SELECT MD5(team_id) AS team_sk, team_id, team_name, conference, division FROM team_lookup
""")
add_pk(DIM_TEAM, "dim_team_pk", ["team_sk"])
print("dim_team rows:", spark.table(DIM_TEAM).count())

dim_team rows: 30


## `dim_date` — generated over the range of games

In [3]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DIM_DATE} (
        date_sk   INT NOT NULL COMMENT 'yyyyMMdd integer key',
        date      DATE,
        year      INT, month INT, day INT,
        day_name  STRING,
        is_weekend BOOLEAN
    ) COMMENT 'Gold dimension - calendar dates spanning the games.'
""")
spark.sql(f"""
    INSERT OVERWRITE {DIM_DATE}
    WITH bounds AS (SELECT MIN(game_date) AS mn, MAX(game_date) AS mx FROM {GAMES}),
    dates AS (
        SELECT explode(sequence((SELECT mn FROM bounds), (SELECT mx FROM bounds), interval 1 day)) AS d
    )
    SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_sk, d AS date,
           year(d) AS year, month(d) AS month, day(d) AS day,
           date_format(d, 'EEEE') AS day_name,
           dayofweek(d) IN (1, 7) AS is_weekend
    FROM dates
""")
add_pk(DIM_DATE, "dim_date_pk", ["date_sk"])
print("dim_date rows:", spark.table(DIM_DATE).count())

dim_date rows: 979


## `fact_games` — one row per game

In [4]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {FACT_GAMES} (
        game_sk STRING NOT NULL, date_sk INT, season INT, game_date DATE, is_playoff BOOLEAN,
        home_team_sk STRING, away_team_sk STRING,
        home_points INT, away_points INT, home_win BOOLEAN, home_margin INT, total_points INT
    )
    CLUSTER BY (season, game_date)
    COMMENT 'Gold fact - one row per game.'
""")
spark.sql(f"""
    INSERT OVERWRITE {FACT_GAMES}
    SELECT g.game_sk, CAST(date_format(g.game_date, 'yyyyMMdd') AS INT) AS date_sk,
           g.season, g.game_date, g.is_playoff,
           ht.team_sk AS home_team_sk, at.team_sk AS away_team_sk,
           g.home_points, g.away_points, g.home_win, g.home_margin, g.total_points
    FROM {GAMES} g
    LEFT JOIN {DIM_TEAM} ht ON g.home_team = ht.team_id
    LEFT JOIN {DIM_TEAM} at ON g.away_team = at.team_id
""")
add_pk(FACT_GAMES, "fact_games_pk", ["game_sk"])
add_fk(FACT_GAMES, "fact_games_date_fk", ["date_sk"], DIM_DATE, ["date_sk"])
add_fk(FACT_GAMES, "fact_games_home_fk", ["home_team_sk"], DIM_TEAM, ["team_sk"])
add_fk(FACT_GAMES, "fact_games_away_fk", ["away_team_sk"], DIM_TEAM, ["team_sk"])
print("fact_games rows:", spark.table(FACT_GAMES).count())

fact_games rows: 3940


## `fact_team_game` — one row per team per game (Four Factors + ratings)

In [5]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {FACT_TG} (
        team_game_sk STRING NOT NULL, game_sk STRING, date_sk INT, season INT, game_date DATE,
        is_playoff BOOLEAN, team_sk STRING, opponent_sk STRING, is_home BOOLEAN, won BOOLEAN, margin INT,
        efg DOUBLE, tov_pct DOUBLE, orb_pct DOUBLE, ft_rate DOUBLE,
        def_efg DOUBLE, def_tov_pct DOUBLE, drb_pct DOUBLE, def_ft_rate DOUBLE,
        off_rtg DOUBLE, def_rtg DOUBLE, net_rtg DOUBLE, poss DOUBLE
    )
    CLUSTER BY (season, game_date)
    COMMENT 'Gold fact - one row per team per game with Four Factors and efficiency ratings.'
""")
spark.sql(f"""
    INSERT OVERWRITE {FACT_TG}
    SELECT t.team_game_sk, t.game_sk, CAST(date_format(t.game_date, 'yyyyMMdd') AS INT) AS date_sk,
           t.season, t.game_date, t.is_playoff,
           tt.team_sk AS team_sk, ot.team_sk AS opponent_sk, t.is_home, t.won, t.margin,
           t.efg, t.tov_pct, t.orb_pct, t.ft_rate, t.def_efg, t.def_tov_pct, t.drb_pct, t.def_ft_rate,
           t.off_rtg, t.def_rtg, t.net_rtg, t.poss
    FROM {TEAM_BOX} t
    LEFT JOIN {DIM_TEAM} tt ON t.team = tt.team_id
    LEFT JOIN {DIM_TEAM} ot ON t.opponent = ot.team_id
""")
add_pk(FACT_TG, "fact_team_game_pk", ["team_game_sk"])
add_fk(FACT_TG, "fact_team_game_game_fk", ["game_sk"], FACT_GAMES, ["game_sk"])
add_fk(FACT_TG, "fact_team_game_team_fk", ["team_sk"], DIM_TEAM, ["team_sk"])
add_fk(FACT_TG, "fact_team_game_date_fk", ["date_sk"], DIM_DATE, ["date_sk"])
print("fact_team_game rows:", spark.table(FACT_TG).count())

fact_team_game rows: 7880


## Serving views

In [6]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {UC_CATALOG}.{GOLD_SCHEMA}.v_team_season_summary AS
    SELECT f.season, dt.team_id, dt.team_name, dt.conference,
           COUNT(*) AS games,
           SUM(CASE WHEN f.won THEN 1 ELSE 0 END) AS wins,
           ROUND(AVG(CASE WHEN f.won THEN 1.0 ELSE 0.0 END), 3) AS win_pct,
           ROUND(AVG(f.efg), 3) AS efg, ROUND(AVG(f.tov_pct), 3) AS tov_pct,
           ROUND(AVG(f.orb_pct), 3) AS orb_pct, ROUND(AVG(f.ft_rate), 3) AS ft_rate,
           ROUND(AVG(f.off_rtg), 1) AS off_rtg, ROUND(AVG(f.def_rtg), 1) AS def_rtg,
           ROUND(AVG(f.net_rtg), 1) AS net_rtg
    FROM {FACT_TG} f JOIN {DIM_TEAM} dt ON f.team_sk = dt.team_sk
    WHERE NOT f.is_playoff
    GROUP BY f.season, dt.team_id, dt.team_name, dt.conference
""")

spark.sql(f"""
    CREATE OR REPLACE VIEW {UC_CATALOG}.{GOLD_SCHEMA}.v_home_court AS
    SELECT season,
           COUNT(*) AS games,
           ROUND(AVG(CASE WHEN home_win THEN 1.0 ELSE 0.0 END), 3) AS home_win_rate,
           ROUND(AVG(home_margin), 2) AS avg_home_margin
    FROM {FACT_GAMES} WHERE NOT is_playoff
    GROUP BY season ORDER BY season
""")

print("Views created. Top net-rating teams (most recent season):")
mx = spark.sql(f"SELECT MAX(season) FROM {FACT_GAMES}").collect()[0][0]
spark.sql(f"""
    SELECT team_name, games, wins, win_pct, off_rtg, def_rtg, net_rtg
    FROM {UC_CATALOG}.{GOLD_SCHEMA}.v_team_season_summary
    WHERE season = {mx} ORDER BY net_rtg DESC LIMIT 10
""").show(truncate=False)

Views created. Top net-rating teams (most recent season):


+----------------------+-----+----+-------+-------+-------+-------+
|team_name             |games|wins|win_pct|off_rtg|def_rtg|net_rtg|
+----------------------+-----+----+-------+-------+-------+-------+
|OKLAHOMA CITY THUNDER |82   |68  |0.829  |117.2  |104.8  |12.4   |
|BOSTON CELTICS        |82   |61  |0.744  |117.5  |108.3  |9.3    |
|CLEVELAND CAVALIERS   |82   |64  |0.780  |118.8  |109.5  |9.3    |
|MINNESOTA TIMBERWOLVES|82   |49  |0.598  |113.8  |108.7  |5.1    |
|LOS ANGELES CLIPPERS  |82   |50  |0.610  |112.6  |107.9  |4.7    |
|MEMPHIS GRIZZLIES     |82   |48  |0.585  |114.6  |110.0  |4.6    |
|HOUSTON ROCKETS       |82   |52  |0.634  |111.9  |107.6  |4.2    |
|NEW YORK KNICKS       |82   |51  |0.622  |115.4  |111.4  |4.0    |
|DENVER NUGGETS        |82   |50  |0.610  |117.0  |113.4  |3.7    |
|GOLDEN STATE WARRIORS |82   |48  |0.585  |112.1  |108.9  |3.2    |
+----------------------+-----+----+-------+-------+-------+-------+



Star schema + views are live. **Next:** `05_build_features` engineers leakage-safe pre-game
features for the model.